# Preparing Training Data

First of all, we need to convert the training data from .json to .spacy (so that we can train our model)

In [ ]:
import re
import spacy
from spacy.tokenizer import Tokenizer

prefix_re = re.compile(r'[.,;:?!(\[\'"</]')
suffix_re = re.compile(r'[.,;:?!)\]\'">■™]')
infix_re = re.compile(r'[-/+=&]')

special_cases = {
    "Ruminococcusgnavusgroup": [{"ORTH": "Ruminococcusgnavus"}, {"ORTH":"group"}],
    "TAMs": [{"ORTH": "TAM"}, {"ORTH": "s"}],
    "SGMs": [{"ORTH": "SGM"}, {"ORTH": "s"}],
}

def custom_tokenizer(nlp):
    tokenizer = Tokenizer(
        nlp.vocab,
        prefix_search=prefix_re.search,
        suffix_search=suffix_re.search,
        infix_finditer=infix_re.finditer
    )


    '''# Add the special cases to tokenizer
    for text, case in special_cases.items():
        tokenizer.add_special_case(text, case)'''

    return tokenizer

In [ ]:
def print_doc_tokens(title, doc, expected_text, label, start, end): # for debugging
    print('-'*60)
    print(title)
    
    tokens_span = []
    for token in doc:
        tokens_span.append((token, token.idx, token.idx + len(token.text)))
    
    print('Tokens -> ', tokens_span)
    print(f'Expected Entity: {expected_text} (Label: {label}, Start: {start}, End: {end})')

def get_spans(entities, doc, title):
    res = []
    for ent in entities:
        start = int(ent['start_idx'])
        end = int(ent['end_idx']) + 1
        label = ent['label']

        span = doc.char_span(start, end, label=label, alignment_mode='strict')
        if span:
            res.append(span)
        else:
            span = doc.char_span(start, end, label=label, alignment_mode='expand')
            if span:
                res.append(span)
                if span.text != ent['text_span']:
                    print(f"Span: {span.text}, Text: {ent['text_span']}")
            else:
                print("Entity not added: ", ent['text_span'])

    # If two entities overlap, keep the one that covers more text
    filtered_ents = []
    for ent in sorted(res, key=lambda e: (e.start, -len(e.text))):  # Sort by start index, prefer longer entities
        if not any(ent.start < e.end and ent.end > e.start for e in filtered_ents):
            filtered_ents.append(ent)
    return filtered_ents

In [ ]:
def prepare_data(json_files):
    db = DocBin()
    for json_file in json_files:
        print('#'*60)
        print('Parsing {}'.format(json_file))
        f = open(json_file)
        data = json.load(f)

        num_ents = 0
        num_parsed_ents = 0

        for article in data:
            title = data[article]['metadata']['title']
            abstract = data[article]['metadata']['abstract']
            entities = data[article]['entities']
    
            title_doc = nlp(title)
            abstract_doc = nlp(abstract)

            title_ents = [ent for ent in entities if ent['location'] == 'title']
            abstract_ents = [ent for ent in entities if ent['location'] == 'abstract']

            num_ents += len(title_ents) + len(abstract_ents)

            title_doc.ents = get_spans(title_ents, title_doc, title)
            abstract_doc.ents = get_spans(abstract_ents, abstract_doc, title)

            num_parsed_ents += len(title_doc.ents) + len(abstract_doc.ents)

            db.add(title_doc)
            db.add(abstract_doc)

        print(f'{num_parsed_ents}/{num_ents}')
        print(f'Articles: {len(data)}')
    return db

In [ ]:
import json
from spacy.tokens import DocBin

train_json = [
    'gutbrainie2025/Annotations/Train/platinum_quality/json_format/train_platinum.json',
    'gutbrainie2025/Annotations/Train/gold_quality/json_format/train_gold.json',
    'gutbrainie2025/Annotations/Train/silver_quality/json_format/train_silver.json',
    'gutbrainie2025/Annotations/Train/bronze_quality/json_format/train_bronze.json'
]

dev_json = [
    'gutbrainie2025/Annotations/Dev/json_format/dev.json'
]

nlp = spacy.blank('en')
nlp.tokenizer = custom_tokenizer(nlp)

train_db = prepare_data(train_json)
dev_db = prepare_data(dev_json)
train_db.to_disk('./train/tok.spacy')
dev_db.to_disk('./dev/tok.spacy')

In [ ]:
!python -m spacy init fill-config base_config.cfg config.cfg
!python -m spacy train config.cfg --output ./out/tok --paths.train ./train/tok.spacy --paths.dev ./dev/tok.spacy

In [ ]:
!python -m spacy evaluate out/base/model-best/ ./dev/base.spacy

In [ ]:
!python -m spacy evaluate out/tok/model-best/ ./dev/tok.spacy

In [ ]:
!python -m spacy evaluate out/tok_special/model-best/ ./dev/tok_special.spacy